# Multiverse Hybrid v3.0 — Stage 7 Final Settlement / OOS

Independent audit APPROVE 後の frozen Stage-7 専用です。**iPhoneでは「ランタイム → すべてのセルを実行」だけ**で構いません。

通常のI/O障害は同一凍結ルールで最大3回自動再試行します。Segment C は final configuration をDriveへ原子的に固定してから一度だけ開きます。巨大ZIP・自動ダウンロードは行いません。


In [ ]:
%pip -q install lxml numpy


In [ ]:

from google.colab import drive
from pathlib import Path
import hashlib, json, shutil, subprocess, time

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(1<<20), b''):
            h.update(c)
    return h.hexdigest()

def copy_exact(src, dst, expected_sha, attempts=3):
    src=Path(src); dst=Path(dst)
    if not src.is_file():
        raise RuntimeError(f"FAIL-CLOSED missing input: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    for n in range(1, attempts+1):
        try:
            if dst.exists():
                dst.unlink()
            print(f"[COPY] {src.name} attempt {n}/{attempts}")
            shutil.copyfile(src, dst)
            obs=sha256_file(dst)
            if obs != expected_sha:
                raise RuntimeError(f"SHA mismatch {dst.name}: {obs} != {expected_sha}")
            print(f"[COPY PASS] {dst.name} sha={obs}")
            return dst
        except Exception as e:
            print(f"[COPY RETRY] {type(e).__name__}: {e}")
            if dst.exists():
                dst.unlink()
            if n == attempts:
                raise
            time.sleep(3)

def run_logged(cmd, log_path, label):
    print(f"▶ {label}")
    p=subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as w:
        w.write(f"\n=== {label} return={p.returncode} ===\n")
        w.write(p.stdout)
        if not p.stdout.endswith("\n"):
            w.write("\n")
    tail="\n".join(p.stdout.splitlines()[-40:])
    print(tail)
    print(f"[RETURN] {p.returncode}")
    return p.returncode

def load_json_if(path):
    p=Path(path)
    if not p.is_file():
        return None
    try:
        return json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        return None

def main():
    drive.mount('/content/drive')
    MY=Path('/content/drive/MyDrive')
    OUT=MY/'MULTIVERSE_ALL_MARKET_STAGE7_SETTLEMENT_EVAL_v1'
    OUT.mkdir(parents=True, exist_ok=True)
    FINAL=OUT/'STAGE7_FINAL_OOS_RECEIPT_v1.json'
    ABREC=OUT/'STAGE7_AB_SELECTION_RECEIPT_v1.json'
    LOG=OUT/'STAGE7_FINAL_ORCHESTRATOR_LOG_v1.txt'

    existing=load_json_if(FINAL)
    if existing and existing.get('status')=='COMPLETE':
        print('✅ STAGE7 FINAL ALREADY COMPLETE — NO RERUN')
        print(json.dumps({
            'final_configuration_id':existing.get('final_configuration_id'),
            'oos_verdict':existing.get('oos_verdict'),
            'segment_C_metrics':existing.get('segment_C_metrics'),
            'bootstrap':existing.get('bootstrap'),
            'ECON_HOLDOUT1000':existing.get('ECON_HOLDOUT1000'),
        }, ensure_ascii=False, indent=2))
        return

    abr=load_json_if(ABREC)
    if abr and abr.get('status') in {'NO_A_ELIGIBLE_CONFIGURATION','NO_B_VALIDATED_CONFIGURATION'}:
        print('✅ STAGE7 SCIENTIFIC TERMINAL HALT — SEGMENT C UNTOUCHED')
        print(json.dumps(abr, ensure_ascii=False, indent=2))
        return

    REPO=Path('/content/multiverse-stage7-final')
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.check_call(['git','clone','--depth','1','https://github.com/fufufu1116/multiverse-research.git',str(REPO)])

    EXPECTED={
      'v3/historical_all_market/stage7_frozen_evaluator_v2.py':'ce8e109fa4c20f683ea1ec999b2fe5dd6f49c865',
      'v3/historical_all_market/stage7_settlement_bulk_runner_v1.py':'b258ced7250071d642ebf2d40127146b22e9a313',
      'v3/historical_all_market/kdreams_settlement_recovery_v1.py':'b8b8ab0e0904541bd6fc45e7fe415d323e63ec45',
      'v3/historical_all_market/stage456_preoutcome_decision_engine_v1.py':'a0ed6984969b0b98af1b074ef9fd2348f16604a0',
      'v3/historical_all_market/governance/STAGE3_TICKET_FILTER_FAMILY_PREREG_v1.md':'ba4175bb044bcacfa66a7b8d089e92c04762b2e6',
      'v3/historical_all_market/governance/STAGE4_CONSENSUS_AGREEMENT_GATE_PREREG_v1.md':'f5bb38e97dd2543842308f9b8ee401957d2e5216',
      'v3/historical_all_market/governance/STAGE5_PORTFOLIO_TEMPLATE_PREREG_v1.md':'f13b5aa5584d260d30032c269cfc205a312f2426',
      'v3/historical_all_market/governance/STAGE6_BANKROLL_RISK_POLICY_PREREG_v1.md':'7dc0ac09440755ad1c43959237c0d975be11b245',
      'v3/historical_all_market/governance/STAGE7_TIME_SPLIT_SELECTION_VALIDATION_PREREG_v1.md':'0cb70520777d4ac9d00ddd90b888df1f403c3a7e',
      'v3/historical_all_market/governance/STAGE7_EXECUTION_CONVENTIONS_FREEZE_v1.md':'b388ef5622d4c92ae4df96ad0105882b4994adf4',
      'v3/historical_all_market/governance/INDEPENDENT_GOVERNANCE_STAGE7_SETTLEMENT_APPROVE_RECEIPT_v1.json':'71e87740ded33ea73c3f534d39830080ad8b43bb',
    }
    for rel,exp in EXPECTED.items():
        obs=subprocess.check_output(['git','-C',str(REPO),'hash-object',rel], text=True).strip()
        if obs != exp:
            raise RuntimeError(f"FAIL-CLOSED Git blob mismatch {rel}: {obs} != {exp}")
    print('✅ EXACT STAGE7 CODE / FREEZE / AUDIT BINDINGS PASS')

    SRC_STAGE2=MY/'MULTIVERSE_ALL_MARKET_STAGE2_PRICE_EV_v1'/'DEV2000_ALL_MARKET_PRICE_EV_CATALOG_v1.jsonl'
    SRC_PRED=MY/'MULTIVERSE_DEV2000_PREDICTION_LOCK_v3_IPHONE_LITE'/'DEV2000_CANDIDATE_A_B1A_RECONSTITUTED_v1_PREDICTIONS.csv'
    SRC_UNIV=MY/'MULTIVERSE_DEV2000_UNIVERSE_RECOVERY'/'DEV2000_UNIVERSE_v1.csv'
    PROV=MY/'MULTIVERSE_DEV2000_RESULT_COLLECTION_v3_HARDENED'/'DEV2000_RESULT_PROVENANCE_v3.jsonl'
    RAW=MY/'MULTIVERSE_DEV2000_RESULT_COLLECTION_v3_HARDENED'/'RAW_RESULT_QUARANTINE'
    if not PROV.is_file() or not RAW.is_dir():
        raise RuntimeError('FAIL-CLOSED DEV2000 provenance/raw quarantine missing')

    LOCAL=Path('/content/multiverse-stage7-input')
    STAGE2=copy_exact(SRC_STAGE2, LOCAL/'stage2.jsonl', '34ad32bed6e8b4d700864c46f4533bef1da254c7d87dc7ffe6ec266fd74530dc')
    PRED=copy_exact(SRC_PRED, LOCAL/'predictions.csv', '772eca4d26f177b94a86ccf7c1b8486e3cdbac0cae454d76ce91fadeca5f1d51')
    UNIV=copy_exact(SRC_UNIV, LOCAL/'universe.csv', 'eb561c9cad5121cf689b237d44a08d089f375a2b2b728e34e91a48338446f3b1')

    SETT=OUT/'SETTLEMENT_ONLY'
    BULKREC=OUT/'STAGE7_SETTLEMENT_BULK_RECEIPT_v1.json'
    needed=[SETT/'DEV2000_SETTLEMENT_A_v1.jsonl',SETT/'DEV2000_SETTLEMENT_B_v1.jsonl',SETT/'DEV2000_SETTLEMENT_C_UNTOUCHED_v1.jsonl',SETT/'STAGE7_SETTLEMENT_BULK_QUALITY_v1.json']
    bulk=load_json_if(BULKREC)
    bulk_ok=bool(bulk and bulk.get('status')=='PASS_COMPLETE' and bulk.get('ECON_HOLDOUT1000')=='SEALED' and all(x.is_file() for x in needed))
    if not bulk_ok:
        runner=REPO/'v3/historical_all_market/stage7_settlement_bulk_runner_v1.py'
        for attempt in range(1,4):
            rc=run_logged(['python',str(runner),'--mydrive',str(MY),'--repo-root',str(REPO)], LOG, f'SETTLEMENT_BULK attempt {attempt}/3')
            bulk=load_json_if(BULKREC)
            if rc==0 and bulk and bulk.get('status')=='PASS_COMPLETE' and all(x.is_file() for x in needed):
                bulk_ok=True
                break
            time.sleep(3)
    else:
        print('✅ SETTLEMENT BULK ALREADY PASS — REUSE')

    if not bulk_ok:
        fatal=OUT/'STAGE7_SETTLEMENT_BULK_FATAL_v1.json'
        if fatal.is_file():
            print(fatal.read_text(encoding='utf-8'))
        raise RuntimeError('FAIL-CLOSED Settlement bulk did not reach PASS after 3 identical technical attempts')

    evaluator=REPO/'v3/historical_all_market/stage7_frozen_evaluator_v2.py'
    common=[
        '--repo-root',str(REPO),
        '--stage2-jsonl',str(STAGE2),
        '--prediction-csv',str(PRED),
        '--universe-csv',str(UNIV),
        '--settlement-dir',str(SETT),
        '--out-dir',str(OUT),
    ]

    freeze=OUT/'FINAL_DEV2000_CONFIGURATION_FREEZE_v1.json'
    abr=load_json_if(ABREC)
    ab_pass=bool(abr and abr.get('status')=='PASS_FINAL_CONFIG_FROZEN' and freeze.is_file())
    if not ab_pass:
        for attempt in range(1,4):
            rc=run_logged(['python',str(evaluator),'--phase','ab',*common], LOG, f'STAGE7_AB attempt {attempt}/3')
            abr=load_json_if(ABREC)
            if abr and abr.get('status') in {'NO_A_ELIGIBLE_CONFIGURATION','NO_B_VALIDATED_CONFIGURATION'}:
                print('✅ STAGE7 SCIENTIFIC TERMINAL HALT — SEGMENT C UNTOUCHED')
                print(json.dumps(abr, ensure_ascii=False, indent=2))
                return
            if rc==0 and abr and abr.get('status')=='PASS_FINAL_CONFIG_FROZEN' and freeze.is_file():
                ab_pass=True
                break
            time.sleep(3)
    else:
        print('✅ STAGE7 A/B ALREADY PASS — FINAL CONFIG REUSE')

    if not ab_pass:
        raise RuntimeError('FAIL-CLOSED A/B selection did not reach a frozen final configuration after 3 identical technical attempts')

    final=load_json_if(FINAL)
    if not (final and final.get('status')=='COMPLETE'):
        for attempt in range(1,4):
            rc=run_logged(['python',str(evaluator),'--phase','c',*common], LOG, f'STAGE7_C_SINGLE_SHOT attempt {attempt}/3')
            final=load_json_if(FINAL)
            if rc==0 and final and final.get('status')=='COMPLETE':
                break
            time.sleep(3)

    final=load_json_if(FINAL)
    if not (final and final.get('status')=='COMPLETE'):
        raise RuntimeError('FAIL-CLOSED frozen Segment C / bootstrap did not complete after 3 identical technical attempts; same frozen trial only')

    print('\n✅ STAGE7 FINAL COMPLETE')
    print(json.dumps({
        'final_configuration_id':final.get('final_configuration_id'),
        'configuration':final.get('configuration'),
        'oos_verdict':final.get('oos_verdict'),
        'segment_B_selected_metrics':final.get('segment_B_selected_metrics'),
        'segment_C_metrics':final.get('segment_C_metrics'),
        'bootstrap':final.get('bootstrap'),
        'scientific_segment_c_scoring_count':final.get('scientific_segment_c_scoring_count'),
        'post_c_rescue_tuning_performed':final.get('post_c_rescue_tuning_performed'),
        'model_refit_performed':final.get('model_refit_performed'),
        'ECON_HOLDOUT1000':final.get('ECON_HOLDOUT1000'),
        'holdout_access':final.get('holdout_access'),
    }, ensure_ascii=False, indent=2))

main()
